<a href="https://colab.research.google.com/github/samikshanimje/SmartECG-HD/blob/main/notebooks/06_dashboard_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/SmartECG-HD"

INFERENCE_DIR = os.path.join(PROJECT_ROOT, "inference")

DECISION_DIR = os.path.join(PROJECT_ROOT, "decision")

os.makedirs(DECISION_DIR, exist_ok=True)

In [3]:
with open(
    os.path.join(INFERENCE_DIR,"prediction.json"),
    "r"
) as f:

    prediction = json.load(f)

prediction

{'prediction': 'N',
 'confidence': 99.9857177734375,
 'probabilities': {'F': 1.6447880625491962e-05,
  'N': 99.9857177734375,
  'Q': 7.404028292512521e-05,
  'S': 0.014179657213389874,
  'V': 6.984045285207685e-06}}

In [4]:
predicted_class = prediction["prediction"]

confidence = prediction["confidence"]

probabilities = prediction["probabilities"]

print(predicted_class)

print(confidence)

N
99.9857177734375


In [5]:
disease_flags = {

    "Arrhythmia": False,

    "Atrial Fibrillation": False,

    "PVC": False,

    "Bradycardia": False,

    "Tachycardia": False,

    "ST Abnormality": False,

    "Myocardial Infarction": False

}

In [6]:
if predicted_class == "N":

    disease_flags["Arrhythmia"] = False

elif predicted_class == "V":

    disease_flags["Arrhythmia"] = True

    disease_flags["PVC"] = True

elif predicted_class == "S":

    disease_flags["Arrhythmia"] = True

    disease_flags["Atrial Fibrillation"] = True

elif predicted_class == "F":

    disease_flags["Arrhythmia"] = True

elif predicted_class == "Q":

    disease_flags["Arrhythmia"] = True

In [7]:
risk_score = round(

    100 - confidence,

    2

)

if predicted_class != "N":

    risk_score = min(

        100,

        risk_score + 40

    )

risk_score

0.01

In [8]:
if predicted_class == "N":

    recommendation = (
        "No immediate cardiac abnormality detected. Continue routine monitoring."
    )

elif predicted_class == "V":

    recommendation = (
        "Possible Premature Ventricular Contractions detected. Cardiology consultation recommended."
    )

elif predicted_class == "S":

    recommendation = (
        "Possible Supraventricular abnormality detected. Further ECG evaluation recommended."
    )

elif predicted_class == "F":

    recommendation = (
        "Fusion beats detected. Clinical review advised."
    )

else:

    recommendation = (
        "Abnormal ECG detected. Further investigation recommended."
    )

In [9]:
report = {

    "prediction": predicted_class,

    "confidence": confidence,

    "risk_score": risk_score,

    "disease_flags": disease_flags,

    "recommendation": recommendation

}

report

{'prediction': 'N',
 'confidence': 99.9857177734375,
 'risk_score': 0.01,
 'disease_flags': {'Arrhythmia': False,
  'Atrial Fibrillation': False,
  'PVC': False,
  'Bradycardia': False,
  'Tachycardia': False,
  'ST Abnormality': False,
  'Myocardial Infarction': False},
 'recommendation': 'No immediate cardiac abnormality detected. Continue routine monitoring.'}

In [10]:
with open(

    os.path.join(
        DECISION_DIR,
        "risk_report.json"
    ),

    "w"

) as f:

    json.dump(
        report,
        f,
        indent=4
    )

print("✅ Report Saved")

✅ Report Saved


In [11]:
pd.DataFrame(

    disease_flags.items(),

    columns=["Disease","Detected"]

).to_csv(

    os.path.join(
        DECISION_DIR,
        "disease_flags.csv"
    ),

    index=False
)

print("✅ Disease Table Saved")

✅ Disease Table Saved


In [12]:
print("="*50)

for file in os.listdir(DECISION_DIR):

    print(file)

print("="*50)

risk_report.json
disease_flags.csv
